In [ ]:
import hail as hl
hl.init(spark_conf={'spark.driver.memory': '32g'})
from hail.plot import show
from pprint import pprint
import pandas as pd
import numpy as np
hl.plot.output_notebook()
rg_new = hl.ReferenceGenome.from_fasta_file(fasta_file='/path/to/local_data/references/human_g1k_v37_fixedChr.fasta',
                                            name="hg19",
                                            index_file='/path/to/local_data/references/human_g1k_v37_fixedChr.fasta.fai')

In [ ]:
# Continue with Hail after setting up the final DF. Parsing results, calculating AF, etc.
# get gene out as a separate field 
# additional funcotation information
# AF in each cohort
# AF in gnomad or 1kg
result = hl.read_matrix_table('/path/to/local_data/EGA_recall/output-joint/hail_filtered.mt')

In [ ]:
func_list = result.rows().info.FUNCOTATION.collect()
func_df = pd.DataFrame([a[0].replace("[", "").replace("]", "")  for a in func_list])[0].str.split("|", expand=True)
func_df.columns=["Gencode_34_hugoSymbol","Gencode_34_ncbiBuild","Gencode_34_chromosome","Gencode_34_start","Gencode_34_end","Gencode_34_variantClassification","Gencode_34_secondaryVariantClassification","Gencode_34_variantType","Gencode_34_refAllele","Gencode_34_tumorSeqAllele1","Gencode_34_tumorSeqAllele2","Gencode_34_genomeChange","Gencode_34_annotationTranscript","Gencode_34_transcriptStrand","Gencode_34_transcriptExon","Gencode_34_transcriptPos","Gencode_34_cDnaChange","Gencode_34_codonChange","Gencode_34_proteinChange","Gencode_34_gcContent","Gencode_34_referenceContext","Gencode_34_otherTranscripts","ACMGLMMLof_LOF_Mechanism","ACMGLMMLof_Mode_of_Inheritance","ACMGLMMLof_Notes","ACMG_recommendation_Disease_Name","ClinVar_VCF_AF_ESP","ClinVar_VCF_AF_EXAC","ClinVar_VCF_AF_TGP","ClinVar_VCF_ALLELEID","ClinVar_VCF_CLNDISDB","ClinVar_VCF_CLNDISDBINCL","ClinVar_VCF_CLNDN","ClinVar_VCF_CLNDNINCL","ClinVar_VCF_CLNHGVS","ClinVar_VCF_CLNREVSTAT","ClinVar_VCF_CLNSIG","ClinVar_VCF_CLNSIGCONF","ClinVar_VCF_CLNSIGINCL","ClinVar_VCF_CLNVC","ClinVar_VCF_CLNVCSO","ClinVar_VCF_CLNVI","ClinVar_VCF_DBVARID","ClinVar_VCF_GENEINFO","ClinVar_VCF_MC","ClinVar_VCF_ORIGIN","ClinVar_VCF_RS","ClinVar_VCF_SSR","ClinVar_VCF_ID","ClinVar_VCF_FILTER","LMMKnown_LMM_FLAGGED","LMMKnown_ID","LMMKnown_FILTER"]
func_df.index = result.locus.collect()
result = result.annotate_rows(gene_symbol=list(func_df['Gencode_34_hugoSymbol']))

In [ ]:
# split out by disease
res_ET = hl.variant_qc(result.filter_cols(result.col.pheno.study == "ET"))
res_MF = hl.variant_qc(result.filter_cols(result.col.pheno.study == "PMF"))
res_PV = hl.variant_qc(result.filter_cols(result.col.pheno.study == "PV"))



In [ ]:
# a=[x[0] if x!=None else 0 for x in res_ET.variant_qc.AF.collect()]
# b=[x[0] if x!=None else 0 for x in res_PV.variant_qc.AF.collect()]
# np.corrcoef(a, b)


In [ ]:
pd.options.display.max_columns = None
display(func_df)

In [ ]:
# build a way to query all the variants
# get query variants from the hail object
chroms = [a.contig[3:] for a in list(result.rows().locus.collect())]
poss = [a.position for a in list(result.rows().locus.collect())]
refs = [a[0] for a in  result.rows().alleles.collect()]
alts = [a[1] for a in  result.rows().alleles.collect()]
df_variants_query = pd.DataFrame({"chrom":chroms, "pos":poss, "ref":refs, "alt":alts})
# Do query in a separate env because of incompatible packages
df_variants_query.to_csv("/path/to/local_data/EGA_recall/df_variants_query.tsv", sep="\t")

In [ ]:
# read back in results
df_AF = pd.read_csv("/path/to/local_data/EGA_recall/df_AF.tsv", sep='\t')

In [ ]:
res_ET = res_ET.annotate_rows(gnomad_AF=list(df_AF["AF"]))
res_PV = res_PV.annotate_rows(gnomad_AF=list(df_AF["AF"]))
res_MF = res_MF.annotate_rows(gnomad_AF=list(df_AF["AF"]))


In [ ]:
# clinvar data
import gzip
f= '/path/to/local_data/gnomad_db/variant_summary.txt.gz'
df_clinvar = pd.read_csv(gzip.GzipFile(f), sep='\t', header=0)


In [ ]:
df_clinvar.head()
df_clinvar["key"] = df_clinvar["Chromosome"].astype('str') + "_" + df_clinvar["PositionVCF"].astype('str') + "_" + df_clinvar["ReferenceAlleleVCF"] + "_" + df_clinvar["AlternateAlleleVCF"]
df_clinvar = df_clinvar.set_index("key")

In [ ]:
df_variants_query["key"] = df_variants_query["chrom"].astype('str') + "_" + df_variants_query["pos"].astype('str') + "_" + df_variants_query["ref"] + "_" + df_variants_query["alt"]
df_variants_query = df_variants_query.set_index("key")

In [ ]:
func_df["chr"] = [a[3:] for a in func_df["Gencode_34_chromosome"].astype('str')]
func_df["key"] = func_df["chr"].astype('str') + "_" + func_df["Gencode_34_start"].astype('str') + "_" + func_df["Gencode_34_refAllele"] + "_" + func_df["Gencode_34_tumorSeqAllele2"]
func_df = func_df.set_index("key")

In [ ]:
func_df["Gencode_34_start"].astype('str')

In [ ]:
# 97680 variants contained in clinVar
print(len(set(df_clinvar.index.values).intersection(df_variants_query.index.values)))
df_clinvar_sub = df_clinvar.loc[list(set(df_clinvar.index.values).intersection(df_variants_query.index.values)),]

In [ ]:
# prepare a dataframe for each disease
df_final_ET = df_variants_query
df_final_PV = df_variants_query
df_final_MF = df_variants_query


In [ ]:
df_final_ET["gomad_AF"] = df_AF["AF"].values
df_final_ET["gene_symbol"] = func_df["Gencode_34_hugoSymbol"].values
df_final_ET["ET_AF"] = [a[1] if a is not None else np.nan for a in res_ET.variant_qc.AF.collect()]
df_final_ET["call_rate"] = res_ET.variant_qc.call_rate.collect()

df_final_PV["gomad_AF"] = df_AF["AF"].values
df_final_PV["gene_symbol"] = func_df["Gencode_34_hugoSymbol"].values
df_final_PV["PV_AF"] = [a[1] if a is not None else np.nan for a in res_PV.variant_qc.AF.collect()]
df_final_PV["call_rate"] = res_PV.variant_qc.call_rate.collect()

df_final_MF["gomad_AF"] = df_AF["AF"].values
df_final_MF["gene_symbol"] = func_df["Gencode_34_hugoSymbol"].values
df_final_MF["MF_AF"] = [a[1] if a is not None else np.nan for a in res_MF.variant_qc.AF.collect()]
df_final_MF["call_rate"] = res_MF.variant_qc.call_rate.collect()

df_final_ET_filtered = df_final_ET.loc[df_final_ET["call_rate"] > 0.5]
df_final_PV_filtered = df_final_PV.loc[df_final_PV["call_rate"] > 0.5]
df_final_MF_filtered = df_final_MF.loc[df_final_MF["call_rate"] > 0.5]


In [ ]:
from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row 
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, LinearColorMapper, ColorBar
output_notebook()

print(df_final_ET_filtered.shape)
p = figure(title = "High-quality germline variants: ET")
color_mapper = LinearColorMapper(palette='Magma256', low=0.5, high=1)

p.scatter('gomad_AF','ET_AF',source=df_final_ET_filtered, color={'field': 'call_rate', 'transform': color_mapper}, fill_alpha=0.1, size=2)
p.xaxis.axis_label = "gnomAD AF"
p.yaxis.axis_label = "AF in ET population"
color_bar = ColorBar(color_mapper=color_mapper,
                     location=(0, 0))
p.add_layout(color_bar, 'right')
show(p)

In [ ]:
print(df_final_PV_filtered.shape)
p = figure(title = "High-quality germline variants: PV")
color_mapper = LinearColorMapper(palette='Magma256', low=0.5, high=1)

p.scatter('gomad_AF','PV_AF',source=df_final_PV_filtered, color={'field': 'call_rate', 'transform': color_mapper}, fill_alpha=0.1, size=2)
p.xaxis.axis_label = "gnomAD AF"
p.yaxis.axis_label = "AF in PV population"
color_bar = ColorBar(color_mapper=color_mapper,
                     location=(0, 0))
p.add_layout(color_bar, 'right')
show(p)

In [ ]:
print(df_final_MF_filtered.shape)
p = figure(title = "High-quality germline variants: MF")
color_mapper = LinearColorMapper(palette='Magma256', low=0.5, high=1)

p.scatter('gomad_AF','MF_AF',source=df_final_MF_filtered, color={'field': 'call_rate', 'transform': color_mapper}, fill_alpha=0.1, size=2)
p.xaxis.axis_label = "gnomAD AF"
p.yaxis.axis_label = "AF in MF population"
color_bar = ColorBar(color_mapper=color_mapper,
                     location=(0, 0))
p.add_layout(color_bar, 'right')
show(p)

In [ ]:
# need to get genes of interest 
HALLMARK_DNA_REPAIR = ["AAAS", "ADA", "ADCY6", "ADRM1", "AGO4", "AK1", "AK3", "ALYREF", "APRT", "ARL6IP1", "BCAM", "BCAP31", "BOLA2", "BRF2", "CANT1", "CCNO", "CDA", "CETN2", "CLP1", "CMPK2", "COX17", "CSTF3", "DAD1", "DCTN4", "DDB1", "DDB2", "DGCR8", "DGUOK", "DUT", "EDF1", "EIF1B", "ELL", "ELOA", "ERCC1", "ERCC2", "ERCC3", "ERCC4", "ERCC5", "ERCC8", "FEN1", "GMPR2", "GPX4", "GSDME", "GTF2A2", "GTF2B", "GTF2F1", "GTF2H1", "GTF2H3", "GTF2H5", "GTF3C5", "GUK1", "HCLS1", "HPRT1", "IMPDH2", "ITPA", "LIG1", "MPC2", "MPG", "MRPL40", "NCBP2", "NELFB", "NELFCD", "NELFE", "NFX1", "NME1", "NME3", "NME4", "NPR2", "NT5C", "NT5C3A", "NUDT21", "NUDT9", "PCNA", "PDE4B", "PDE6G", "PNP", "POLA1", "POLA2", "POLB", "POLD1", "POLD3", "POLD4", "POLE4", "POLH", "POLL", "POLR1C", "POLR1D", "POLR1H", "POLR2A", "POLR2C", "POLR2D", "POLR2E", "POLR2F", "POLR2G", "POLR2H", "POLR2I", "POLR2J", "POLR2K", "POLR3C", "POLR3GL", "POM121", "PRIM1", "RAD51", "RAD52", "RAE1", "RALA", "RBX1", "REV3L", "RFC2", "RFC3", "RFC4", "RFC5", "RNMT", "RPA2", "RPA3", "RRM2B", "SAC3D1", "SDCBP", "SEC61A1", "SF3A3", "SMAD5", "SNAPC4", "SNAPC5", "SRSF6", "SSRP1", "STX3", "SUPT4H1", "SUPT5H", "SURF1", "TAF10", "TAF12", "TAF13", "TAF1C", "TAF6", "TAF9", "TARBP2", "TK2", "TMED2", "TP53", "TSG101", "TYMS", "UMPS", "UPF3B", "USP11", "VPS28", "VPS37B", "VPS37D", "XPC", "ZNF707", "ZWINT"]
KEGG_HOMOLOGOUS_RECOMBINATION = ["BLM", "BRCA2", "EME1", "MRE11", "MUS81", "NBN", "POLD1", "POLD2", "POLD3", "POLD4", "RAD50", "RAD51", "RAD51B", "RAD51C", "RAD51D", "RAD52", "RAD54B", "RAD54L", "RPA1", "RPA2", "RPA3", "RPA4", "SEM1", "SSBP1", "TOP3A", "TOP3B", "XRCC2", "XRCC3"]
KEGG_DNA_REPLICATION = ["DNA2", "FEN1", "LIG1", "MCM2", "MCM3", "MCM4", "MCM5", "MCM6", "MCM7", "PCNA", "POLA1", "POLA2", "POLD1", "POLD2", "POLD3", "POLD4", "POLE", "POLE2", "POLE3", "POLE4", "PRIM1", "PRIM2", "RFC1", "RFC2", "RFC3", "RFC4", "RFC5", "RNASEH1", "RNASEH2A", "RNASEH2B", "RNASEH2C", "RPA1", "RPA2", "RPA3", "RPA4", "SSBP1"]

gene_list1 = [
    HALLMARK_DNA_REPAIR,
    KEGG_HOMOLOGOUS_RECOMBINATION,
    KEGG_DNA_REPLICATION
]
gene_list = list(set([item for sublist in gene_list1 for item in sublist]))
print(len(gene_list))

In [ ]:
# Merge with clinvar data
df_final_ET_filtered2 = pd.merge(df_final_ET_filtered, df_clinvar_sub, how="outer", left_index=True, right_index=True).iloc[:,0:17]
df_final_ET_filtered2["ClinicalSignificance"] = df_final_ET_filtered2["ClinicalSignificance"].astype('str')
df_final_ET_filtered3 = pd.merge(df_final_ET_filtered2, func_df, how="outer", left_index=True, right_index=True).iloc[:,0:36]
# Remove columns
df_final_ET_filtered3 = df_final_ET_filtered3.drop(["PV_AF", "MF_AF", "Gencode_34_hugoSymbol", "Gencode_34_ncbiBuild", "Gencode_34_chromosome", "Gencode_34_start", "Gencode_34_end", "Gencode_34_refAllele", "Gencode_34_tumorSeqAllele1", "Gencode_34_tumorSeqAllele2"], axis=1)
# Filter to gene list of interest
df_final_ET_filtered_genes = df_final_ET_filtered3.loc[df_final_ET_filtered3["gene_symbol"].isin(gene_list)]
df_final_ET_filtered_genes = df_final_ET_filtered_genes[~np.isnan(df_final_ET_filtered_genes["gomad_AF"]).values]
# Differenc in AF compared to gnomad
df_final_ET_filtered_genes["AF_difference"] = df_final_ET_filtered_genes["ET_AF"] -df_final_ET_filtered_genes["gomad_AF"] 
df_final_ET_filtered_genes["AF_difference_abs"] = abs(df_final_ET_filtered_genes["ET_AF"] -df_final_ET_filtered_genes["gomad_AF"] )
df_final_ET_filtered_genes = df_final_ET_filtered_genes.loc[df_final_ET_filtered_genes["ET_AF"] >0]
# Sorting criteria
df_final_ET_filtered_genes.sort_values(by=["AF_difference_abs"], axis=0, inplace=True, ascending=False)
df_final_ET_filtered_genes.to_csv("/path/to/local_data/EGA_recall/df_final_ET_filtered_genes.tsv", sep="\t")
print(df_final_ET_filtered_genes.shape)
df_final_ET_filtered_genes.head()

In [ ]:
# Merge with clinvar data
df_final_PV_filtered2 = pd.merge(df_final_PV_filtered, df_clinvar_sub, how="outer", left_index=True, right_index=True).iloc[:,0:17]
df_final_PV_filtered2["ClinicalSignificance"] = df_final_PV_filtered2["ClinicalSignificance"].astype('str')
df_final_PV_filtered3 = pd.merge(df_final_PV_filtered2, func_df, how="outer", left_index=True, right_index=True).iloc[:,0:36]
# Remove columns
df_final_PV_filtered3 = df_final_PV_filtered3.drop(["ET_AF", "MF_AF", "Gencode_34_hugoSymbol", "Gencode_34_ncbiBuild", "Gencode_34_chromosome", "Gencode_34_start", "Gencode_34_end", "Gencode_34_refAllele", "Gencode_34_tumorSeqAllele1", "Gencode_34_tumorSeqAllele2"], axis=1)
# Filter to gene list of interest
df_final_PV_filtered_genes = df_final_PV_filtered3.loc[df_final_PV_filtered3["gene_symbol"].isin(gene_list)]
df_final_PV_filtered_genes = df_final_PV_filtered_genes[~np.isnan(df_final_PV_filtered_genes["gomad_AF"]).values]
# Differenc in AF compared to gnomad
df_final_PV_filtered_genes["AF_difference"] = df_final_PV_filtered_genes["PV_AF"] -df_final_PV_filtered_genes["gomad_AF"] 
df_final_PV_filtered_genes["AF_difference_abs"] = abs(df_final_PV_filtered_genes["PV_AF"] -df_final_PV_filtered_genes["gomad_AF"] )
df_final_PV_filtered_genes = df_final_PV_filtered_genes.loc[df_final_PV_filtered_genes["PV_AF"] >0]
# Sorting criteria
df_final_PV_filtered_genes.sort_values(by=["AF_difference_abs"], axis=0, inplace=True, ascending=False)w
df_final_PV_filtered_genes.to_csv("/path/to/local_data/EGA_recall/df_final_PV_filtered_genes.tsv", sep="\t")
print(df_final_PV_filtered_genes.shape)
df_final_PV_filtered_genes.head()

In [ ]:
# Merge with clinvar data
df_final_MF_filtered2 = pd.merge(df_final_MF_filtered, df_clinvar_sub, how="outer", left_index=True, right_index=True).iloc[:,0:17]
df_final_MF_filtered2["ClinicalSignificance"] = df_final_MF_filtered2["ClinicalSignificance"].astype('str')
df_final_MF_filtered3 = pd.merge(df_final_MF_filtered2, func_df, how="outer", left_index=True, right_index=True).iloc[:,0:36]
# Remove columns
df_final_MF_filtered3 = df_final_MF_filtered3.drop(["ET_AF", "PV_AF", "Gencode_34_hugoSymbol", "Gencode_34_ncbiBuild", "Gencode_34_chromosome", "Gencode_34_start", "Gencode_34_end", "Gencode_34_refAllele", "Gencode_34_tumorSeqAllele1", "Gencode_34_tumorSeqAllele2"], axis=1)
# Filter to gene list of interest
df_final_MF_filtered_genes = df_final_MF_filtered3.loc[df_final_MF_filtered3["gene_symbol"].isin(gene_list)]
df_final_MF_filtered_genes = df_final_MF_filtered_genes[~np.isnan(df_final_MF_filtered_genes["gomad_AF"]).values]
# Differenc in AF compared to gnomad
df_final_MF_filtered_genes["AF_difference"] = df_final_MF_filtered_genes["MF_AF"] -df_final_MF_filtered_genes["gomad_AF"] 
df_final_MF_filtered_genes["AF_difference_abs"] = abs(df_final_MF_filtered_genes["MF_AF"] -df_final_MF_filtered_genes["gomad_AF"] )
df_final_MF_filtered_genes = df_final_MF_filtered_genes.loc[df_final_MF_filtered_genes["MF_AF"] >0]
# Sorting criteria
df_final_MF_filtered_genes.sort_values(by=["AF_difference_abs"], axis=0, inplace=True, ascending=False)
df_final_MF_filtered_genes.to_csv("/path/to/local_data/EGA_recall/df_final_MF_filtered_genes.tsv", sep="\t")
print(df_final_MF_filtered_genes.shape)
df_final_MF_filtered_genes.head()